# Lab 04 · ReAct: choose a tool, then decide

**Prepared by Mustafa Assaf** · 5–10 minute example

## Setup
Run this cell once. It loads the reusable helpers; the example follows below. Practice answers are scripted. For a live model, save `FREEINFERENCE_API_KEY` in Colab Secrets and change `MODE` to `"live"`.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit']):
    %pip install -q "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/aebb52597ab982f0b8b2f0832815b73dd6aca416.zip"
from coursekit import make_ask
MODE = "practice"
ask = make_ask(mode=MODE)

## Let the model choose its next action

Lab 03 called a tool directly. Now the model chooses `read_metrics` or `finish`; Python validates that choice and limits the loop to two turns. The observation is kept in memory for the next call.

This small ReAct-style pattern alternates action and observation, then gives an architecture decision. We use a local version of the metric reader to keep the control flow visible. A complete ReAct agent can choose among more tools.

In [ ]:
rule = 'Choose microservices only if team_size >= 8 AND independent_deploys is true; otherwise choose modular monolith.'
def read_metrics(): return {"team_size": 3, "independent_deploys": False}
observation = ""
for turn in range(2):
    prompt = f"Choose read_metrics or finish. Finish only after reading metrics. Observation: {observation}. Return only the action."
    action = ask(prompt, example="read_metrics" if not observation else "finish").strip()
    print("Action:", action)
    if action == "read_metrics": observation = str(read_metrics())
    elif action == "finish" and observation: break
    else: raise ValueError("Unknown action or missing observation")
else: raise RuntimeError("Turn limit reached before finish")
answer = ask(f"{rule} Metrics: {observation}. Return only the choice.", example="modular monolith").strip().lower()
print("Decision:", answer)
assert answer == "modular monolith", "Check the observation and policy."

**Expected practice sequence:** `read_metrics → finish → modular monolith`. The check evaluates the final decision. An invalid action, an early finish, or an exhausted turn limit fails visibly.

**Small exercise:** make the metric reader return team size 8 and independent deployment `True`. Predict the new answer; update its practice fixture and final assertion, or run live.

A fixed **chain** would always read, choose, then check. A **router** could send a decision to a specialist. Use an agent loop when choosing the next action is useful; extra calls alone do not ensure quality.

## Read further

- [ReAct](https://arxiv.org/abs/2210.03629)
- [Workflows and agentic patterns](https://www.anthropic.com/research/building-effective-agents)